In [ ]:
import sys
!{sys.executable} -m pip install -q easyocr matplotlib requests pytz opencv-python pillow

In [ ]:
import os
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

from database import criar_tabelas, cadastrar_veiculo, listar_moradores,\
    verificar_acesso, registrar_log, obter_historico
from detector import detectar_placa
from utils import extrair_placa, tipo_placa

In [ ]:
criar_tabelas()

print("--- Cadastros Iniciais ---")
cadastrar_veiculo("RIO2A18", "Morador Teste Mercosul", "Ap 01")

print("\n--- Baixando imagem de teste ---")
url = "https://www.azulseguros.com.br/wp-content/uploads/2019/04/placa-mercosul.jpg"
local = "assets/placa_teste.png"

try:
    resp = requests.get(url, timeout=10)
    with open(local, "wb") as f:
        f.write(resp.content)
    print("Imagem salva em assets/placa_teste.png")

    print("\n--- Executando OCR ---")
    texto_bruto, img_processada, bbox, img_original = detectar_placa(local)

    print(f"Texto bruto: '{texto_bruto}'")
    if texto_bruto:
        placa_corrigida = extrair_placa(texto_bruto)
        tipo = tipo_placa(placa_corrigida)
        print(f"Placa extraida: '{placa_corrigida}' (tipo: {tipo})")

        status, nome, ap = verificar_acesso(placa_corrigida)
        registrar_log(placa_corrigida, status)

        if status == "Liberado":
            print(f"\nACESSO LIBERADO: {nome} ({ap})")
        else:
            print(f"\nACESSO NEGADO: placa nao autorizada")

        print("\n--- Imagem processada (regiao do OCR) ---")
        if img_processada is not None:
            plt.figure(figsize=(8, 4))
            plt.imshow(img_processada, cmap='gray')
            plt.axis('off')
            plt.show()
    else:
        print("Nenhuma placa detectada.")

    print("\n--- Historico de Acessos ---")
    for placa, data, sts in obter_historico():
        print(f"Placa: {placa} | {data} | {sts}")

except Exception as e:
    print(f"Erro: {e}")